<a href="https://colab.research.google.com/github/manugd1105/AI_challenge_UPF/blob/main/Obtenci%C3%B3n%20del%20conjunto%20de%20train-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTEBOOK 1/2

### Realizado por:

- Iker Lleida
- Santiago de León
- Manuel González

El fin de este notebook es crear los dataframes (que exportaremos como csv) para poder realizar el ML sobre ellos.

En líneas generales, en este dataset se descomprime el zip con los ficheros de audio (que está guardado en drive, aunque también se incluye dentro del repo), convierte los datos a numéricos utilizando los estadísticos de centralización/dispersión de las 12 primeras mfcc, convierte a un dataframe de pandas y guarda en un fichero csv para que se pueda utilizar en el siguiente notebook de python (donde realizaremos el ML) sin tener que volver a correr todo esto.

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Descomprimir el archivo LA (tarda ~ 3 min)
!unzip --q "/content/drive/MyDrive/Colab_Notebooks/Reto de Inteligencia Artificial/DS_10283_3336/LA.zip"

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7787040.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_2924301.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9249366.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_3442936.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7772915.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_5569336.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7773607.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7813281.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9705954.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_2427464.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1000273.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_5263550.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1642109.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1339848.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9495857.flac  
  inflati

In [ ]:
import os

def count_files_in_directory(path):
    if not os.path.exists(path):
        return f"La carpeta '{path}' no existe.", 0
    count = 0
    for root, dirs, files in os.walk(path):
        count += len(files)
    return count

# Paths for LA dataset splits
la_train_path = '/content/LA/ASVspoof2019_LA_train/flac'
la_dev_path = '/content/LA/ASVspoof2019_LA_dev/flac'
la_eval_path = '/content/LA/ASVspoof2019_LA_eval/flac'

# Count files for each split
num_files_la_train = count_files_in_directory(la_train_path)
num_files_la_dev = count_files_in_directory(la_dev_path)
num_files_la_eval = count_files_in_directory(la_eval_path)

print(f"Número de ficheros en la carpeta '{la_train_path}' (train): {num_files_la_train}")
print(f"Número de ficheros en la carpeta '{la_dev_path}' (dev): {num_files_la_dev}")
print(f"Número de ficheros en la carpeta '{la_eval_path}' (eval): {num_files_la_eval}")

Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_train/flac' (train): 25380
Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_dev/flac' (dev): 24986
Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_eval/flac' (eval): 71933


Realmente, solo vamos a guardar los conjuntos de train y eval, aunque después en el ML solo usaremos el de train (subdividiéndolo en train/validacion/test).

## Conversión a datos numéricos

In [ ]:
# Instalamos librosa para procesamiento de audio
!pip install librosa soundfile

import os
import librosa
import numpy as np

# Cuantos registros quiero coger para mi dataframe (25380 el total en train)
# Para la carpeta 'eval' hay 71933 ficheros, vamos a coger todos
num_registros = 71933

# Rutas al dataset LA
# Cambiamos las rutas para usar el protocolo y los archivos de audio de evaluación.
la_protocol_path = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
# Usamos la ruta a los archivos de audio de evaluación.
la_audio_path = '/content/LA/ASVspoof2019_LA_eval/flac'

# Función para extraer características de un archivo de audio
def extract_features(audio_file_path, sr=16000, n_mfcc=13):
    try:
        # Cargar el archivo de audio
        audio, sample_rate = librosa.load(audio_file_path, sr=sr)
        # Extraer MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
        # Transponer para tener el formato (n_mfcc, time_steps)
        return mfccs.T
    except Exception as e:
        print(f"Error procesando {audio_file_path}: {e}")
        return None

# Leemos el archivo de protocolo para obtener una lista de audios y sus etiquetas
sample_audio_files = []
labels = []

with open(la_protocol_path, 'r') as f:
    for i, line in enumerate(f):
        # Ejemplo de línea del protocolo eval: LA_0000 LA_E_2628468 - A06 spoof
        parts = line.strip().split(' ')
        if len(parts) >= 3:
            # El ID del archivo de audio es el segundo elemento en el protocolo del ASVspoof2019 eval.
            file_id = parts[1]
            label = parts[-1]
            full_audio_path = os.path.join(la_audio_path, f'{file_id}.flac')
            if os.path.exists(full_audio_path):
                sample_audio_files.append(full_audio_path)
                labels.append(label)
        if i >= num_registros - 1: # Leer hasta num_registros
            break

if not sample_audio_files:
    print(f"No se encontraron archivos de audio en la ruta: {la_audio_path} o el protocolo está vacío/malformado.")
else:
    print(f"Procesando {len(sample_audio_files)} archivos de audio de ejemplo...")
    # Convertir etiquetas a formato numérico
    # 'bonafide' -> 0, 'spoof' -> 1
    label_mapping = {'bonafide': 0, 'spoof': 1}
    numerical_labels = [label_mapping.get(lbl, -1) for lbl in labels]

    # Extraer características para los archivos de ejemplo
    all_features = []
    for i, audio_file in enumerate(sample_audio_files):
        features = extract_features(audio_file)
        if features is not None:
            all_features.append(features)
            print(f"Archivo: {os.path.basename(audio_file)}, MFCCs shape: {features.shape}, Etiqueta numérica: {numerical_labels[i]}")

    if all_features:
        print("\nCaracterísticas extraídas para los archivos. Cada elemento en 'all_features' es un array numpy de MFCCs para un archivo de audio.")
        print("Puedes apilarlos o rellenarlos para que tengan la misma longitud si vas a entrenar un modelo que requiere entradas de tamaño fijo.")
    else:
        print("No se pudieron extraer características de ningón archivo de audio.")

Se han truncado las últimas 5000 líneas del flujo de salida.
Archivo: LA_E_2009225.flac, MFCCs shape: (47, 13), Etiqueta numérica: 1
Archivo: LA_E_2920570.flac, MFCCs shape: (130, 13), Etiqueta numérica: 1
Archivo: LA_E_6636964.flac, MFCCs shape: (28, 13), Etiqueta numérica: 1
Archivo: LA_E_8249159.flac, MFCCs shape: (86, 13), Etiqueta numérica: 1
Archivo: LA_E_2434688.flac, MFCCs shape: (124, 13), Etiqueta numérica: 1
Archivo: LA_E_8279191.flac, MFCCs shape: (272, 13), Etiqueta numérica: 1
Archivo: LA_E_7136149.flac, MFCCs shape: (87, 13), Etiqueta numérica: 1
Archivo: LA_E_3082798.flac, MFCCs shape: (38, 13), Etiqueta numérica: 1
Archivo: LA_E_3064542.flac, MFCCs shape: (131, 13), Etiqueta numérica: 1
Archivo: LA_E_2587770.flac, MFCCs shape: (114, 13), Etiqueta numérica: 1
Archivo: LA_E_3872590.flac, MFCCs shape: (76, 13), Etiqueta numérica: 1
Archivo: LA_E_8341227.flac, MFCCs shape: (74, 13), Etiqueta numérica: 1
Archivo: LA_E_5796437.flac, MFCCs shape: (56, 13), Etiqueta numérica: 

Vamos a convertirlo a un dataframe de pandas y calcular las medidas de centralización/dispersión de las mfcc.

In [ ]:
import pandas as pd
import os
import numpy as np # Import numpy for statistical operations
import scipy.stats # Import scipy.stats for skewness and kurtosis

# To retrieve the SPEAKER_ID and SYSTEM_ID, we need to re-read the protocol file
# and match the audio file names. la_protocol_path is available from the kernel state.

speaker_ids = []
system_ids = []

# Create a mapping from audio_file_name (e.g., 'LA_D_1047731') to its protocol info
protocol_map = {}
with open(la_protocol_path, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')
        if len(parts) >= 5: # Ensure all expected parts are present
            speaker_id = parts[0]
            audio_file_name = parts[1]
            system_id = parts[3] # SYSTEM_ID is the 4th element (index 3)
            # The 5th element (index 4) is the KEY, which corresponds to our 'labels' list
            protocol_map[audio_file_name] = {'speaker_id': speaker_id, 'system_id': system_id}

# Crear una lista de diccionarios para cada entrada
data = []
for i in range(len(sample_audio_files)):
    file_name_with_ext = os.path.basename(sample_audio_files[i])
    file_id_without_ext = os.path.splitext(file_name_with_ext)[0] # e.g., 'LA_D_1047731'

    speaker_id = None
    system_id = None
    if file_id_without_ext in protocol_map:
        info = protocol_map[file_id_without_ext]
        speaker_id = info['speaker_id']
        system_id = info['system_id']

    # Get mfcc_features for the current audio file
    current_mfcc_features = all_features[i]

    # Calculate statistical summaries (mean and std) for each MFCC coefficient
    # over the temporal windows (axis=0)
    mfcc_means = np.mean(current_mfcc_features, axis=0)
    mfcc_stds = np.std(current_mfcc_features, axis=0)
    mfcc_medians = np.median(current_mfcc_features, axis=0)
    mfcc_mins = np.min(current_mfcc_features, axis=0)
    mfcc_maxs = np.max(current_mfcc_features, axis=0)
    mfcc_skewness = scipy.stats.skew(current_mfcc_features, axis=0)
    mfcc_kurtosis = scipy.stats.kurtosis(current_mfcc_features, axis=0)

    entry = {
        'audio_file': sample_audio_files[i],
        'speaker_id': speaker_id,
        'system_id': system_id,
        'label_numerical': numerical_labels[i], # Renaming 'label' to 'label_numerical' for clarity
        'label_text': labels[i], # Add the original text label for clarity (bonafide/spoof)
        'num_temporal_windows': current_mfcc_features.shape[0] # Keep this for context
    }

    # Add mean MFCCs as individual columns
    for j in range(len(mfcc_means)):
        entry[f'mfcc_mean_{j}'] = mfcc_means[j]

    # Add standard deviation MFCCs as individual columns
    for j in range(len(mfcc_stds)):
        entry[f'mfcc_std_{j}'] = mfcc_stds[j]

    # Add median MFCCs as individual columns
    for j in range(len(mfcc_medians)):
        entry[f'mfcc_median_{j}'] = mfcc_medians[j]

    # Add min MFCCs as individual columns
    for j in range(len(mfcc_mins)):
        entry[f'mfcc_min_{j}'] = mfcc_mins[j]

    # Add max MFCCs as individual columns
    for j in range(len(mfcc_maxs)):
        entry[f'mfcc_max_{j}'] = mfcc_maxs[j]

    # Add skewness MFCCs as individual columns
    for j in range(len(mfcc_skewness)):
        entry[f'mfcc_skew_{j}'] = mfcc_skewness[j]

    # Add kurtosis MFCCs as individual columns
    for j in range(len(mfcc_kurtosis)):
        entry[f'mfcc_kurt_{j}'] = mfcc_kurtosis[j]

    data.append(entry)

# Convertir la lista de diccionarios en un DataFrame de pandas
df_features = pd.DataFrame(data)

print("DataFrame de características creado:")
display(df_features.head())
print(f"\nDimensiones del DataFrame: {df_features.shape}")
print(f"Columnas del DataFrame: {df_features.columns.tolist()}")

DataFrame de características creado:


,audio_file,speaker_id,system_id,label_numerical,label_text,num_temporal_windows,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,...,mfcc_kurt_3,mfcc_kurt_4,mfcc_kurt_5,mfcc_kurt_6,mfcc_kurt_7,mfcc_kurt_8,mfcc_kurt_9,mfcc_kurt_10,mfcc_kurt_11,mfcc_kurt_12
0,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_283...,LA_0039,A11,1,spoof,45,-173.321274,99.674423,-18.967428,28.347494,...,-0.125496,-1.175945,-1.015942,-0.913028,-0.936737,-0.919556,-0.359033,0.681294,-0.602497,0.434048
1,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_887...,LA_0014,A14,1,spoof,125,-281.757721,84.629829,-39.956604,21.229477,...,0.108241,-0.488111,-0.792298,-0.438702,-0.455414,-0.855074,0.155858,-0.424825,0.011414,-0.109712
2,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_682...,LA_0040,A16,1,spoof,49,-274.346069,89.168678,-4.030763,26.393579,...,0.082035,-0.991445,0.026241,-0.334211,-0.716008,0.140134,-0.429711,-0.355633,0.235064,-0.809780
3,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_697...,LA_0022,A09,1,spoof,62,-260.305847,96.839836,-35.881580,41.725830,...,-0.823853,-0.606110,-1.260264,-1.154528,-0.698799,-0.808773,-0.075343,-0.491189,-0.546873,1.291234
4,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_593...,LA_0031,A13,1,spoof,182,-182.082596,73.088120,1.822410,22.063999,...,-0.785015,-0.654107,-0.301525,-0.461822,0.033093,-0.506384,-0.377746,1.206588,0.000747,-0.443290



Dimensiones del DataFrame: (71237, 97)
Columnas del DataFrame: ['audio_file', 'speaker_id', 'system_id', 'label_numerical', 'label_text', 'num_temporal_windows', 'mfcc_mean_0', 'mfcc_mean_1', 'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4', 'mfcc_mean_5', 'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8', 'mfcc_mean_9', 'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12', 'mfcc_std_0', 'mfcc_std_1', 'mfcc_std_2', 'mfcc_std_3', 'mfcc_std_4', 'mfcc_std_5', 'mfcc_std_6', 'mfcc_std_7', 'mfcc_std_8', 'mfcc_std_9', 'mfcc_std_10', 'mfcc_std_11', 'mfcc_std_12', 'mfcc_median_0', 'mfcc_median_1', 'mfcc_median_2', 'mfcc_median_3', 'mfcc_median_4', 'mfcc_median_5', 'mfcc_median_6', 'mfcc_median_7', 'mfcc_median_8', 'mfcc_median_9', 'mfcc_median_10', 'mfcc_median_11', 'mfcc_median_12', 'mfcc_min_0', 'mfcc_min_1', 'mfcc_min_2', 'mfcc_min_3', 'mfcc_min_4', 'mfcc_min_5', 'mfcc_min_6', 'mfcc_min_7', 'mfcc_min_8', 'mfcc_min_9', 'mfcc_min_10', 'mfcc_min_11', 'mfcc_min_12', 'mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2', 'mf

Tenemos las columnas:

- speaker_id: ID del hablante, con formato LA_****. Es quien ha hablado o a quien se está intentando imitar con el deepfake.

- system_id: ID of the speech spoofing system (A01 - A19),  or, for bonafide speech SYSTEM-ID is left blank ('-')

- label: 0 es bonafide (genuino) y 1 es spoof (fake).

In [ ]:
df_features

,audio_file,speaker_id,system_id,label_numerical,label_text,num_temporal_windows,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,...,mfcc_kurt_3,mfcc_kurt_4,mfcc_kurt_5,mfcc_kurt_6,mfcc_kurt_7,mfcc_kurt_8,mfcc_kurt_9,mfcc_kurt_10,mfcc_kurt_11,mfcc_kurt_12
0,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_283...,LA_0039,A11,1,spoof,45,-173.321274,99.674423,-18.967428,28.347494,...,-0.125496,-1.175945,-1.015942,-0.913028,-0.936737,-0.919556,-0.359033,0.681294,-0.602497,0.434048
1,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_887...,LA_0014,A14,1,spoof,125,-281.757721,84.629829,-39.956604,21.229477,...,0.108241,-0.488111,-0.792298,-0.438702,-0.455414,-0.855074,0.155858,-0.424825,0.011414,-0.109712
2,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_682...,LA_0040,A16,1,spoof,49,-274.346069,89.168678,-4.030763,26.393579,...,0.082035,-0.991445,0.026241,-0.334211,-0.716008,0.140134,-0.429711,-0.355633,0.235064,-0.809780
3,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_697...,LA_0022,A09,1,spoof,62,-260.305847,96.839836,-35.881580,41.725830,...,-0.823853,-0.606110,-1.260264,-1.154528,-0.698799,-0.808773,-0.075343,-0.491189,-0.546873,1.291234
4,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_593...,LA_0031,A13,1,spoof,182,-182.082596,73.088120,1.822410,22.063999,...,-0.785015,-0.654107,-0.301525,-0.461822,0.033093,-0.506384,-0.377746,1.206588,0.000747,-0.443290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71232,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_166...,LA_0004,-,0,bonafide,76,-393.605957,61.175426,-10.567280,24.054108,...,-0.377455,-0.685722,-0.492614,-0.994269,-0.676583,-0.639860,-0.229282,2.771902,0.112861,0.317184
71233,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_508...,LA_0038,A09,1,spoof,227,-248.028076,89.653305,-9.501684,39.576061,...,-0.981897,0.571156,-0.621770,-0.422124,-0.815093,-0.157820,-0.730911,-0.733601,-0.362555,-0.805651
71234,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_492...,LA_0012,A16,1,spoof,123,-233.477448,60.590111,-16.905754,0.990290,...,-0.158967,-0.834957,-0.059350,-0.916874,-0.346782,-0.002863,-0.420363,-0.909911,1.390054,-0.613701
71235,/content/LA/ASVspoof2019_LA_eval/flac/LA_E_289...,LA_0052,-,0,bonafide,123,-272.191162,38.375851,-7.030939,55.639526,...,-1.281670,-0.553232,-0.186457,-1.290728,-0.985484,-0.601053,-1.319067,-0.942968,-1.091245,-0.019295


In [ ]:
df_features.system_id.unique()

array(['A11', 'A14', 'A16', 'A09', 'A13', '-', 'A12', 'A18', 'A15', 'A08',
       'A17', 'A10', 'A07', 'A19'], dtype=object)

In [ ]:
df_features.to_csv('df_test_manu.csv', index=False)